# 05 -- EAD (exposure at default)

**What this notebook does (plain English):** **EAD** is simply how much money was
still owed on a loan at the moment it defaulted -- the amount truly at risk. For
a mortgage this is just the outstanding balance, because a mortgage is fully
drawn on day one. (Contrast a credit card, which has an undrawn limit the
borrower can run up before defaulting -- that needs an extra "credit conversion
factor"; a mortgage does not.)

**Headline result:** average exposure at default is around **$190k**, and it is
broadly similar across vintages -- EAD is a balance, not a risk gauge.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and keep defaulted loans (EAD is the balance at default).
import pandas as pd
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')
defaulted = base[base['ever_default']].copy()
print('defaulted loans:', len(defaulted))

defaulted loans: 11756


In [3]:
# EAD summary: average and spread of exposure at default, by vintage.
ead_summary = defaulted.groupby('vintage_year').agg(
    defaults=('ead', 'size'),
    mean_ead=('ead', 'mean'),
    median_ead=('ead', 'median'),
    p95_ead=('ead', lambda s: s.quantile(0.95)),
    total_ead=('ead', 'sum'),
).reset_index().round(2)
save_csv(ead_summary, 'output/05_ead_summary.csv')
ead_summary

,vintage_year,defaults,mean_ead,median_ead,p95_ead,total_ead
0,2007,6870,186177.02,168815.41,378126.33,1.279036e+09
1,2008,3677,198213.99,177327.80,399580.18,7.288328e+08
2,2015,1209,197949.91,172449.07,395785.90,2.393214e+08


**Why there is no CCF here:** A credit conversion factor models how much of
an *undrawn* limit a borrower draws before defaulting. A term mortgage has no
undrawn limit -- the full principal is advanced at closing and only ever
amortises down -- so EAD is just the outstanding balance and **no CCF/drawdown
modelling applies**. Knowing CCF belongs to *revolving* products (like a credit
card) and deliberately *not* using it here is the point, not a gap.